# K8s Pod-Size Scheduler, Scoring Simulation

In [103]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [104]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import random
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from main import *

## Define services & run strategies

In [105]:
SEED = 42
services = make_example_services()

svc_df = pd.DataFrame([
    {"service": s.name, "cpu_request": s.cpu_request, "replicas": s.replicas}
    for s in services
])
print(f"Total pods to schedule: {svc_df['replicas'].sum()}")
print(f"Total CPU requested:    {(svc_df['cpu_request'] * svc_df['replicas']).sum()}")
svc_df

Total pods to schedule: 144
Total CPU requested:    634.0


,service,cpu_request,replicas
0,nginx,1.0,23
1,auth,2.0,11
2,api,3.0,7
3,search,4.0,8
4,payment,5.0,3
5,recommendation,6.0,4
6,video,10.0,9
7,inference,14.0,4
8,pipeline,20.0,3
9,analytics,8.0,3


In [106]:
def run_simulation(scorer, label: str, seed: int = SEED) -> tuple[Cluster, str]:
    random.seed(seed)
    nodes = make_nodes(count=10, cpu=88.0)
    cluster = Cluster(nodes, scorer)
    deploy_order = list(services)
    random.shuffle(deploy_order)
    for svc in deploy_order:
        cluster.deploy_service(svc)
    return cluster, label


# Random baseline
class RandomScorer:
    def score(self, node: Node, pod_cpu: float) -> float:
        return random.random()
    def score_all(self, nodes: list[Node], pod_cpu: float) -> list[float]:
        return [self.score(n, pod_cpu) for n in nodes]

In [107]:
clusters: dict[str, Cluster] = {}
for scorer, label in [
    (SoftBucketScorer(), "Soft-Bucket"),
    (DensityScorer(sigma=2.0), "Density"),
    (RandomScorer(), "Random"),
]:
    c, l = run_simulation(scorer, label)
    clusters[l] = c
    print(f"[{l}] scheduled {len(c.history)} pods")

[Soft-Bucket] scheduled 144 pods
[Density] scheduled 144 pods
[Random] scheduled 144 pods


## 1. Node CPU utilization per strategy

In [108]:
rows = []
for label, cluster in clusters.items():
    for n in cluster.nodes.values():
        rows.append({
            "strategy": label,
            "node": n.name,
            "used_cpu": n.used_cpu,
            "free_cpu": n.free_cpu,
            "pod_count": len(n.pods),
        })
util_df = pd.DataFrame(rows)

fig = px.bar(
    util_df, x="node", y="used_cpu", color="strategy",
    barmode="group", title="CPU Utilization per Node",
    labels={"used_cpu": "Used CPU", "node": "Node"},
)
fig.add_hline(y=88, line_dash="dash", line_color="red", annotation_text="Capacity")
fig.update_layout(height=450)
fig.show()

## 2. Pod placement map, every pod on every node

Each rectangle = one pod, height = CPU request, color = service.

In [109]:
def build_placement_df(cluster: Cluster) -> pd.DataFrame:
    rows = []
    for node in cluster.nodes.values():
        y_offset = 0.0
        for pod in sorted(node.pods, key=lambda p: p.cpu_request):
            rows.append({
                "node": node.name,
                "pod": pod.name,
                "service": pod.service,
                "cpu": pod.cpu_request,
                "y_start": y_offset,
                "y_end": y_offset + pod.cpu_request,
            })
            y_offset += pod.cpu_request

    return pd.DataFrame(rows)


def plot_placement(cluster: Cluster, title: str):
    df = build_placement_df(cluster)
    if df.empty:
        return
    
    # Colors per service
    all_services = sorted(df["service"].unique())
    colors = px.colors.qualitative.Set3 + px.colors.qualitative.Pastel
    svc_colors = {s: colors[i % len(colors)] for i, s in enumerate(all_services)}
    
    fig = go.Figure()
    
    # Group by service for legend
    for svc in all_services:
        sdf = df[df["service"] == svc]
        for _, row in sdf.iterrows():
            fig.add_trace(go.Bar(
                x=[row["node"]],
                y=[row["cpu"]],
                base=[row["y_start"]],
                name=svc,
                marker_color=svc_colors[svc],
                legendgroup=svc,
                showlegend=bool(sdf.index[0] == row.name),
                hovertext=f"{row['pod']}<br>{row['cpu']} CPU",
                hoverinfo="text",
                width=0.7,
            ))
    
    fig.add_hline(y=88, line_dash="dash", line_color="red", annotation_text="88 CPU capacity")
    fig.update_layout(
        barmode="stack",
        title=title,
        xaxis_title="Node",
        yaxis_title="CPU",
        height=550,
        legend_title="Service",
    )
    fig.show()


for label, cluster in clusters.items():
    plot_placement(cluster, f"Pod Placement, {label} scheduler")

## 3. Pod-size distribution per node (histogram)

Are similarly-sized pods spread across nodes?

In [110]:
def pod_size_strip(cluster: Cluster, title: str):
    rows = []
    for node in cluster.nodes.values():
        for pod in node.pods:
            rows.append({"node": node.name, "cpu": pod.cpu_request, "service": pod.service})
    df = pd.DataFrame(rows)
    fig = px.strip(
        df, x="node", y="cpu", color="service",
        title=title,
        labels={"cpu": "Pod CPU request", "node": "Node"},
        height=450,
    )
    fig.update_traces(marker_size=8, jitter=0.3)
    fig.show()

for label, cluster in clusters.items():
    pod_size_strip(cluster, f"Pod sizes per node, {label}")

## 4. Soft-bucket mass heatmap

Shows the soft mass per anchor (S/M/L/XL) per node. Ideally all rows look similar.

In [111]:
bucket_labels = [f"{name} ({int(mu)})" for name, (mu, _, _) in
                 zip(["Small", "Mid", "Large", "XL"], DEFAULT_ANCHORS)]

def soft_bucket_heatmap(cluster: Cluster, title: str):
    node_names = sorted(cluster.nodes.keys())
    matrix = []
    for nn in node_names:
        counts = compute_soft_counts(cluster.nodes[nn], DEFAULT_ANCHORS)
        matrix.append(counts)
    arr = np.array(matrix)
    
    fig = go.Figure(go.Heatmap(
        z=arr,
        x=bucket_labels,
        y=node_names,
        colorscale="YlOrRd",
        text=np.round(arr, 2),
        texttemplate="%{text}",
    ))
    fig.update_layout(title=title, height=400, xaxis_title="Bucket", yaxis_title="Node")
    fig.show()

for label, cluster in clusters.items():
    soft_bucket_heatmap(cluster, f"Soft-bucket mass — {label}")

## 5. Step-by-step scoring trace

In [112]:
sb_cluster = clusters["Soft-Bucket"]
score_rows = []
for evt in sb_cluster.history:
    for node_name, sc in evt.scores.items():
        score_rows.append({
            "step": evt.step,
            "pod": evt.pod.name,
            "cpu": evt.pod.cpu_request,
            "node": node_name,
            "score": sc,
            "chosen": node_name == evt.chosen_node,
        })
scores_df = pd.DataFrame(score_rows)

fig = px.scatter(
    scores_df, x="node", y="score", color="chosen",
    animation_frame="step",
    hover_data=["pod", "cpu"],
    title="Soft-Bucket scores per node at each scheduling step",
    labels={"score": "Score S_j", "node": "Node", "chosen": "Chosen?"},
    color_discrete_map={True: "green", False: "gray"},
    height=500,
    range_y=[0, -30]
)
fig.update_traces(marker_size=12)
fig.update_layout(
    xaxis=dict(categoryorder="category ascending"),
    yaxis_title="Score (higher = better)",
)
fig.show()

## 6. Imbalance summary

Standard deviation of per-node CPU and per-node soft-bucket masses, lower = more balanced.

In [113]:
summary_rows = []
for label, cluster in clusters.items():
    cpus = [n.used_cpu for n in cluster.nodes.values()]
    pod_counts = [len(n.pods) for n in cluster.nodes.values()]
    
    # Soft-bucket mass std per bucket
    all_counts = np.array([compute_soft_counts(n, DEFAULT_ANCHORS) for n in cluster.nodes.values()])
    bucket_stds = all_counts.std(axis=0)
    
    summary_rows.append({
        "strategy": label,
        "cpu_std": round(np.std(cpus), 2),
        "pod_count_std": round(np.std(pod_counts), 2),
        "small_std": round(bucket_stds[0], 2),
        "mid_std": round(bucket_stds[1], 2),
        "large_std": round(bucket_stds[2], 2),
        "xl_std": round(bucket_stds[3], 2),
        "total_bucket_std": round(bucket_stds.sum(), 2),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,strategy,cpu_std,pod_count_std,small_std,mid_std,large_std,xl_std,total_bucket_std
0,Soft-Bucket,5.47,0.80,0.38,0.31,0.16,0.30,1.14
1,Density,8.15,0.80,0.38,0.24,0.46,0.32,1.39
2,Random,17.30,2.76,2.52,1.01,1.49,0.59,5.60
